# Qwen3.5 2B LoRA — v5 portable training with 10-step checkpoints

This notebook trains the reviewed **general-domain v5 r1 requirements extractor**. It writes a complete, hash-attested checkpoint to Google Drive every **10 optimizer steps** and refuses an accidental restart when `MINIMUM_RESUME_STEP` is positive.

A 10-step interval reduces how much progress can be lost. It does **not** make each checkpoint smaller: exact resume requires the adapter, optimizer, scheduler, trainer state, and RNG state. Expect substantial Drive usage. The transfer cell packages the latest verified checkpoint into one portable `.tar` file for another Google account.

Run the cells in order. For a fresh run leave `MINIMUM_RESUME_STEP = 0`. After transferring `checkpoint-N`, set it to `N`; training stops before loading weights if that checkpoint is missing or corrupt. Never manufacture a checkpoint from a progress-bar number. The sealed final test is never loaded.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys, tarfile, tempfile

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

RUN_NAME = 'qwen35-2b-lora-v5-portable-10step-r1'
RUN = Path('/content/drive/MyDrive/FYP-model-runs') / RUN_NAME
TRANSFER_DIR = Path('/content/drive/MyDrive/FYP-model-transfer')
RUN.mkdir(parents=True, exist_ok=True)
TRANSFER_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_INTERVAL = 10
# Fresh run: 0. After moving checkpoint-N to another account: N.
MINIMUM_RESUME_STEP = 0
# Destination account only: set this to the transferred .tar path.
IMPORT_BUNDLE = None
# IMPORT_BUNDLE = '/content/drive/MyDrive/FYP-model-transfer/qwen35-2b-lora-v5-portable-10step-r1-step-70.tar'

PROJECT = Path('/content/slm-v5-portable')
FPY = Path('/content/fpy-v5-portable')
REPO_COMMIT = '7469e8b516474d684f5859f1e9a6a09078af944b'
FPY_COMMIT = '04d52c015d1e3ecdefe92b87116f209361509b4b'
MODEL = 'Qwen/Qwen3.5-2B'
MODEL_REVISION = '15852e8c16360a2fea060d615a32b45270f8a8fc'
EXPECTED_MANIFEST_SHA256 = '7e3c6f83bd39a73a386dbbf5e2a7932c45b68af46eaf6641e56474f2cc9dd021'
MAX_LENGTH = 3584

try:
    gpu = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True
    ).strip()
except Exception:
    gpu = 'not available (transfer/export is still available; training requires CUDA)'
print('GPU:', gpu)
print('Run:', RUN)
print('Checkpoint interval:', CHECKPOINT_INTERVAL)
print('Required resume floor:', MINIMUM_RESUME_STEP)
print('Import bundle:', IMPORT_BUNDLE)

In [ ]:
# Optional safe import on a different Google account. Run before setup/data cells.
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if IMPORT_BUNDLE:
    bundle = Path(IMPORT_BUNDLE)
    assert bundle.is_file(), f'Missing transfer bundle: {bundle}'
    existing = list(RUN.iterdir())
    assert not existing, f'Import destination must be empty; found: {[p.name for p in existing]}'
    with tempfile.TemporaryDirectory(dir='/content') as temporary:
        staging = Path(temporary)
        with tarfile.open(bundle, 'r') as archive:
            for member in archive.getmembers():
                target = (staging / member.name).resolve()
                assert str(target).startswith(str(staging.resolve()) + os.sep), member.name
                assert not (member.issym() or member.islnk()), member.name
            archive.extractall(staging, filter='data')
        transfer_manifest = json.loads((staging / 'transfer-manifest.json').read_text())
        for relative, expected in transfer_manifest['files'].items():
            path = staging / relative
            assert path.is_file(), f'Missing bundled file: {relative}'
            assert sha256(path) == expected['sha256'], f'Hash mismatch: {relative}'
        for item in staging.iterdir():
            if item.name != 'transfer-manifest.json':
                shutil.move(str(item), str(RUN / item.name))
    print('Imported and hash-verified:', bundle)
else:
    print('No transfer bundle selected.')

In [ ]:
import json, subprocess, sys
from pathlib import Path

def command(*args):
    print('Running:', *args)
    subprocess.run([str(x) for x in args], check=True)

if not (PROJECT / '.git').is_dir():
    command('git', 'clone', 'https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git', PROJECT)
command('git', '-C', PROJECT, 'fetch', 'origin', 'experiment/v5-grounded-20260919')
command('git', '-C', PROJECT, 'checkout', '--detach', REPO_COMMIT)
actual_repo = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_repo == REPO_COMMIT

if not (FPY / '.git').is_dir():
    command('git', 'clone', 'https://github.com/int-abd-5/fpy.git', FPY)
command('git', '-C', FPY, 'fetch', 'origin', FPY_COMMIT)
command('git', '-C', FPY, 'checkout', '--detach', FPY_COMMIT)
actual_fpy = subprocess.check_output(
    ['git', '-C', str(FPY), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_fpy == FPY_COMMIT

command(sys.executable, '-m', 'pip', 'install', '-r', PROJECT / 'training/requirements.txt')
command(sys.executable, '-m', 'pip', 'install', '-e', FPY)

source_lock = {
    'repository_commit': REPO_COMMIT,
    'fpy_commit': FPY_COMMIT,
    'model': MODEL,
    'model_revision': MODEL_REVISION,
    'license': 'Apache-2.0',
    'checkpoint_interval_steps': CHECKPOINT_INTERVAL,
}
lock_path = RUN / 'source-lock.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text()) == source_lock, 'Existing run has a different source lock'
else:
    lock_path.write_text(json.dumps(source_lock, indent=2) + '\n')

with (RUN / 'pip-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)
print(json.dumps(source_lock, indent=2))

In [ ]:
import hashlib, json, shutil, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

CORPUS = PROJECT / 'corpus-v5/v5-20260919-r1'
subprocess.run([
    sys.executable,
    str(PROJECT / 'scripts/verify-corpus-v5.py'),
    '--output',
    str(CORPUS),
], cwd=PROJECT, check=True)

manifest_hash = hashlib.sha256((CORPUS / 'manifest.json').read_bytes()).hexdigest()
assert manifest_hash == EXPECTED_MANIFEST_SHA256, f'Unexpected corpus manifest: {manifest_hash}'

inputs = RUN / 'inputs'
inputs.mkdir(exist_ok=True)
input_hashes = {}
for name in ('train', 'validation'):
    source = CORPUS / 'sft' / f'{name}.jsonl'
    destination = inputs / f'{name}.jsonl'
    if destination.exists():
        assert destination.read_bytes() == source.read_bytes(), f'{name} input changed'
    else:
        shutil.copy2(source, destination)
    input_hashes[name] = hashlib.sha256(destination.read_bytes()).hexdigest()

preflight = {
    'corpus_manifest_sha256': manifest_hash,
    'verified_max_length': MAX_LENGTH,
    'length_measurement_source': (
        'v5 gate preflight: train max 3529, validation max 3460; rounded to 3584'
    ),
    'input_sha256': input_hashes,
}
preflight_path = RUN / 'length-preflight.json'
if preflight_path.exists():
    assert json.loads(preflight_path.read_text()) == preflight
else:
    preflight_path.write_text(json.dumps(preflight, indent=2) + '\n')

# This records the user's explicit confirmation. It does not invent an independent review ledger.
approval = {
    'corpus_manifest_sha256': manifest_hash,
    'reviewer': 'Abdullah Usman',
    'review_confirmed_at_utc': datetime.now(timezone.utc).isoformat(),
    'human_review_confirmed_by_user': True,
    'train_sample_approved': True,
    'validation_approved': True,
    'compute_run_authorized_by_user': True,
    'pilot_behavior_approved': False,
    'final_test_authorized': False,
    'protocol_deviation': (
        'User requested immediate full training before completion of the matched pilot behavior comparison.'
    ),
    'claim_limit': (
        'Training authorization only; model selection and paper-quality claims still require matched validation evaluation.'
    ),
}
approval_path = RUN / 'human-review-approval.json'
if approval_path.exists():
    previous = json.loads(approval_path.read_text())
    assert previous['corpus_manifest_sha256'] == manifest_hash
    assert previous['human_review_confirmed_by_user'] is True
else:
    approval_path.write_text(json.dumps(approval, indent=2) + '\n')

print(json.dumps(preflight, indent=2))
print('Approval recorded:', approval_path)
print('Final sealed test remains untouched.')


In [ ]:
# Mandatory resume audit. This performs full hash verification before training.
sys.path.insert(0, str(PROJECT))
from training.train_lora import checkpoint_step, latest_checkpoint, load_manifest, verify_checkpoint

output = RUN / 'full'
manifest_path = output / 'run-manifest.json'
if manifest_path.is_file():
    manifest = load_manifest(output)
    fingerprint = manifest['run_fingerprint']
    selected = latest_checkpoint(output, fingerprint)
else:
    manifest = None
    fingerprint = None
    selected = None

contents = sorted(p.name for p in output.iterdir()) if output.exists() else []
print('Output contents:', contents)
print('Verified checkpoints:')
if output.exists():
    for path in sorted(output.glob('checkpoint-*'), key=lambda p: checkpoint_step(p)):
        try:
            marker = verify_checkpoint(path, fingerprint)
            print(' ', path.name, marker['completed_at'])
        except Exception as exc:
            print(' INVALID', path.name, type(exc).__name__, str(exc))

if MINIMUM_RESUME_STEP > 0:
    assert selected is not None, f'Expected checkpoint >= {MINIMUM_RESUME_STEP}, but none is verified'
    assert checkpoint_step(selected) >= MINIMUM_RESUME_STEP, (
        f'Latest verified checkpoint is {checkpoint_step(selected)}, expected >= {MINIMUM_RESUME_STEP}'
    )
if contents and selected is None:
    raise RuntimeError('Run folder is occupied but has no verified checkpoint. Use a new RUN_NAME or import the correct bundle.')

print('RESUME DECISION:', f'{selected} (step {checkpoint_step(selected)})' if selected else 'FRESH START AT STEP 0')

In [ ]:
import subprocess, sys, time
from pathlib import Path

gpu_check = subprocess.run(
    ['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False
)
if gpu_check.returncode != 0:
    raise RuntimeError('Training requires a CUDA GPU runtime. Transfer/export cells can run on CPU.')

output = RUN / 'full'
log_path = RUN / 'full-console.log'

args = [
    sys.executable,
    'training/train_lora.py',
    '--model', MODEL,
    '--revision', MODEL_REVISION,
    '--train', str(RUN / 'inputs/train.jsonl'),
    '--validation', str(RUN / 'inputs/validation.jsonl'),
    '--output', str(output),
    '--epochs', '2',
    '--save-steps', str(CHECKPOINT_INTERVAL),
    '--learning-rate', '5e-5',
    '--early-stopping-patience', '0',
    '--max-length', str(MAX_LENGTH),
    '--resume-from-checkpoint', 'auto',
    '--minimum-resume-step', str(MINIMUM_RESUME_STEP),
]

print('Training command:', ' '.join(map(str, args)))
print('Log:', log_path)
start = time.monotonic()
with log_path.open('a', encoding='utf-8', buffering=1) as log:
    log.write('\n=== PORTABLE 10-STEP TRAINING START OR RESUME ===\n')
    process = subprocess.Popen(
        args,
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    return_code = process.wait()
    elapsed = time.monotonic() - start
    log.write(f'\nreturn_code={return_code} elapsed_seconds={elapsed}\n')

print('Elapsed seconds:', elapsed)
print('Exit code:', return_code)
if return_code != 0:
    raise RuntimeError(
        'Training stopped. Preserve the Drive folder, reconnect a GPU, and rerun this same cell to resume.'
    )
print('Training completed. All 10-step durability checkpoints remain in:', output)

In [ ]:
# Export the latest verified checkpoint as one cross-account transfer file.
# This can run after an interruption from a CPU-only Colab runtime.
import io
from datetime import datetime, timezone

output = RUN / 'full'
manifest = load_manifest(output)
selected = latest_checkpoint(output, manifest['run_fingerprint'])
assert selected is not None, 'No complete checkpoint is available to export'
verify_checkpoint(selected, manifest['run_fingerprint'])
step = checkpoint_step(selected)
required_resume_files = {
    'adapter_config.json', 'optimizer.pt', 'scheduler.pt',
    'trainer_state.json', 'training_args.bin', 'rng_state.pth',
    'checkpoint-complete.json',
}
missing_resume_files = sorted(name for name in required_resume_files if not (selected / name).is_file())
assert not missing_resume_files, f'Checkpoint cannot resume exactly; missing: {missing_resume_files}'

relative_paths = []
for relative in [
    Path('source-lock.json'), Path('human-review-approval.json'),
    Path('length-preflight.json'), Path('inputs/train.jsonl'),
    Path('inputs/validation.jsonl'), Path('full/run-manifest.json'),
]:
    if (RUN / relative).is_file():
        relative_paths.append(relative)
for path in sorted(selected.rglob('*')):
    if path.is_file():
        relative_paths.append(path.relative_to(RUN))

transfer_manifest = {
    'schema_version': 1,
    'run_name': RUN_NAME,
    'checkpoint_step': step,
    'run_fingerprint': manifest['run_fingerprint'],
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'files': {},
}
for relative in relative_paths:
    path = RUN / relative
    transfer_manifest['files'][relative.as_posix()] = {
        'bytes': path.stat().st_size, 'sha256': sha256(path),
    }

bundle = TRANSFER_DIR / f'{RUN_NAME}-step-{step}.tar'
temporary_bundle = TRANSFER_DIR / f'.{RUN_NAME}-step-{step}-{os.getpid()}.tar.tmp'
with tarfile.open(temporary_bundle, 'w') as archive:
    payload = (json.dumps(transfer_manifest, indent=2) + '\n').encode()
    info = tarfile.TarInfo('transfer-manifest.json')
    info.size = len(payload)
    archive.addfile(info, io.BytesIO(payload))
    for relative in relative_paths:
        archive.add(RUN / relative, arcname=relative.as_posix(), recursive=False)
temporary_bundle.replace(bundle)

receipt = {
    'bundle': str(bundle), 'bytes': bundle.stat().st_size,
    'sha256': sha256(bundle), 'checkpoint_step': step,
}
bundle.with_suffix('.receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
print(json.dumps(receipt, indent=2))
print(f'On the new account set IMPORT_BUNDLE to this file and MINIMUM_RESUME_STEP = {step}.')

In [ ]:
import hashlib, json
from datetime import datetime, timezone
from pathlib import Path

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

output = RUN / 'full'
best = output / 'best-adapter'
assert best.is_dir(), 'best-adapter was not produced'
required = [best / 'adapter_model.safetensors', best / 'adapter_config.json']
for path in required:
    assert path.is_file(), f'Missing {path}'

checkpoints = sorted(
    p.name
    for p in output.glob('checkpoint-*')
    if p.is_dir() and (p / 'checkpoint-complete.json').is_file()
)
summary = {
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'run': str(RUN),
    'repository_commit': REPO_COMMIT,
    'model': MODEL,
    'model_revision': MODEL_REVISION,
    'corpus_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'epochs_requested': 2,
    'checkpoint_interval_steps': CHECKPOINT_INTERVAL,
    'learning_rate': 5e-5,
    'max_length': MAX_LENGTH,
    'complete_checkpoints': checkpoints,
    'best_adapter_files': {
        path.name: {'bytes': path.stat().st_size, 'sha256': sha256(path)}
        for path in required
    },
    'selection_status': 'pending behavioral validation; final-step is not automatically task-best',
    'final_test_status': 'untouched',
}
summary_path = RUN / 'training-completion-summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('Saved:', summary_path)

## After training

Send `training-completion-summary.json` and the last 150 lines of `full-console.log` back to Codex. The next stage selects the behavioral checkpoint on v5 validation and then runs the structured-output experiments one arm at a time. Do not run the sealed final set during method development.